In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random
import sys
sys.path.append("tic tac toe")

# TicTacToe environment (copied here for self-contained use)
class TicTacToe:
    def __init__(self):
        self.board = [""] * 9
        self.current_player = "X"

    def reset(self):
        self.board = [""] * 9
        self.current_player = "X"
        return self._state()

    def _state(self):
        return [1 if x == "X" else -1 if x == "O" else 0 for x in self.board]

    def available_actions(self):
        return [i for i, x in enumerate(self.board) if x == ""]

    def step(self, action):
        if self.board[action] != "":
            raise ValueError("invalid action")
        self.board[action] = self.current_player
        if self.check_winner():
            return self._state(), 1 if self.current_player == "X" else -1, True
        elif all(x != "" for x in self.board):
            return self._state(), 0, True
        self.current_player = "O" if self.current_player == "X" else "X"
        return self._state(), 0, False

    def check_winner(self):
        win_conditions = [
            [0,1,2],[3,4,5],[6,7,8],
            [0,3,6],[1,4,7],[2,5,8],
            [0,4,8],[2,4,6]
        ]
        for c in win_conditions:
            if self.board[c[0]] == self.board[c[1]] == self.board[c[2]] != "":
                return True
        return False

class QNetwork(nn.Module):
    def __init__(self, state_size, action_size):
        super(QNetwork, self).__init__()
        self.fc1 = nn.Linear(state_size, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, action_size)

    def forward(self, state):
        x = torch.relu(self.fc1(state))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

state_size = 9   # TicTacToe board
action_size = 9  # 9 possible positions
q_network = QNetwork(state_size, action_size)
target_network = QNetwork(state_size, action_size)
target_network.load_state_dict(q_network.state_dict())
optimizer = optim.Adam(q_network.parameters(), lr=0.001)
print("Networks initialized.")

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = []
        self.capacity = capacity

    def push(self, experience):
        self.buffer.append(experience)
        if len(self.buffer) > self.capacity:
            self.buffer.pop(0)

    def sample(self, batch_size):
        return random.sample(self.buffer, batch_size)

    def size(self):
        return len(self.buffer)

replay_buffer = ReplayBuffer(capacity=10000)

In [ ]:
def compute_td_loss(batch, q_network, target_network, gamma=0.99):
    states, actions, rewards, next_states, dones = zip(*batch)
    states      = torch.tensor(states,      dtype=torch.float32)
    actions     = torch.tensor(actions,     dtype=torch.long)
    rewards     = torch.tensor(rewards,     dtype=torch.float32)
    next_states = torch.tensor(next_states, dtype=torch.float32)
    dones       = torch.tensor(dones,       dtype=torch.bool)
    q_values = q_network(states).gather(1, actions.unsqueeze(1))
    next_q_values = target_network(next_states).max(1)[0]
    target_q_values = rewards + gamma * next_q_values * (~dones)
    loss = torch.mean((q_values - target_q_values.unsqueeze(1)) ** 2)
    return loss

def select_action(state, env, epsilon):
    available = env.available_actions()
    if random.random() < epsilon:
        return random.choice(available)
    state_t = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
    q_vals = q_network(state_t).detach().numpy()[0]
    # mask unavailable actions
    mask = np.full(9, -np.inf)
    mask[available] = q_vals[available]
    return int(np.argmax(mask))

# Training loop
episodes = 5000
batch_size = 64
epsilon = 1.0
epsilon_min = 0.1
epsilon_decay = 0.995
target_update_freq = 100

replay_buffer = ReplayBuffer(capacity=10000)
env = TicTacToe()
wins = draws = losses = 0

for episode in range(1, episodes + 1):
    state = env.reset()
    done = False

    while not done:
        action = select_action(state, env, epsilon)
        next_state, reward, done = env.step(action)

        # opponent makes random move
        if not done:
            opp_action = random.choice(env.available_actions())
            next_state, opp_reward, done = env.step(opp_action)
            if done and opp_reward != 0:
                reward = -1  # agent lost

        replay_buffer.push((state, action, reward, next_state, done))
        state = next_state

        if replay_buffer.size() >= batch_size:
            batch = replay_buffer.sample(batch_size)
            loss = compute_td_loss(batch, q_network, target_network)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    if reward == 1:
        wins += 1
    elif reward == 0:
        draws += 1
    else:
        losses += 1

    epsilon = max(epsilon_min, epsilon * epsilon_decay)

    if episode % target_update_freq == 0:
        target_network.load_state_dict(q_network.state_dict())

    if episode % 500 == 0:
        print(f"Episode {episode} | Wins: {wins} Draws: {draws} Losses: {losses} | Epsilon: {epsilon:.3f}")
        wins = draws = losses = 0

print("Training complete!")